# Financial Statement Viewer

Select a filing, pick a statement, visualize it, and save it — all from dropdown menus.

In [8]:
import sys
from collections import defaultdict
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import HTML, clear_output, display

sys.path.insert(0, str(Path("../src").resolve()))
from xbrl_extraction import Document

OUTPUT_DIR = Path("../data/output")

In [9]:
# ── Shared state ────────────────────────────────────────────────────────────
_doc: Document | None = None
_role_map: dict[str, str] = {}   # role_short -> role_definition
_role_short_map: dict[str, str] = {}  # display_label -> role_short

In [ ]:
# ── Helpers ─────────────────────────────────────────────────────────────────

# Pure dimensional plumbing — skip these entirely (don't emit a row, but recurse)
_SKIP_SUFFIXES = ("Table", "Axis", "Domain", "Member", "LineItems", "LineItem")

# Section-header nodes — emit a header row but no value
_HEADER_SUFFIXES = ("Abstract",)

def _is_skip(concept: str) -> bool:
    return concept.split(":", 1)[-1].endswith(_SKIP_SUFFIXES)

def _is_abstract(concept: str) -> bool:
    return concept.split(":", 1)[-1].endswith(_HEADER_SUFFIXES)

# Tables that indicate a supplemental sub-entity schedule — these are
# the only tables whose declared axis/member scopes the entire fact lookup.
# StatementTable is the primary financial statement container and is NEVER
# treated as a scope filter, even when it declares a single member.
_SUPPLEMENTAL_TABLE_PREFIXES = (
    "ScheduleOfCondensed",
    "Condensed",
)

def _is_supplemental_table(table_name: str) -> bool:
    local = table_name.split(":")[-1]
    return any(local.startswith(p) for p in _SUPPLEMENTAL_TABLE_PREFIXES)

def _infer_primary_dims(doc, role_short):
    """
    Return {axis: member} only when the role is a supplemental sub-entity
    schedule (e.g. parent-company-only condensed financial statements).

    Rule: an axis scopes the table when ALL of these hold:
      1. It hangs off a *supplemental* table (Condensed* or ScheduleOfCondensed*).
         StatementTable — used by primary financial statements — is excluded.
      2. It has exactly one leaf member (a scope filter, not a breakdown).
         Axes with 2+ leaf members create within-row breakdowns and are excluded.

    Returns {} for primary statements and any role not matching both conditions.
    """
    arcs = [a for a in doc.pres.arcs if a.role_short == role_short]
    cbp  = defaultdict(list)
    for a in arcs:
        cbp[a.parent].append(a.child)
    all_nodes = {a.parent for a in arcs} | {a.child for a in arcs}
    # Only consider supplemental tables — exclude StatementTable
    tables = [
        n for n in all_nodes
        if n.split(":")[-1].endswith("Table") and _is_supplemental_table(n)
    ]
    primary = {}
    for table in tables:
        for axis in cbp.get(table, []):
            if not axis.split(":")[-1].endswith("Axis"):
                continue
            members = []
            def collect(node, _m=members):
                for child in cbp.get(node, []):
                    if child.split(":")[-1].endswith("Member"):
                        _m.append(child)
                    collect(child)
            collect(axis)
            leaf = [m for m in members if not cbp.get(m)]
            if len(leaf) == 1:
                primary[axis] = leaf[0]
    return primary

def _values_at(doc, period_end, required_dims=None):
    """
    Return concept -> authoritative value for period_end.

    Standard mode (required_dims is None):
      Plain (no-dimension) facts are the consolidated primary figures and
      always win. Dimensioned facts are used only as fallback when no plain
      fact exists for a concept+period.

    Scoped mode (required_dims provided):
      For supplemental sub-entity schedules (e.g. parent-company-only).
      Facts carrying the required dimension tags override plain facts,
      because every line item in the table belongs to that sub-entity scope.
      Plain facts still fill gaps where no scoped fact exists.
    """
    matching = {cid for cid, ctx in doc.periods.items()
                if (ctx.type == "instant" and ctx.date == period_end)
                or (ctx.type == "duration" and ctx.end  == period_end)}
    plain  = {}
    scoped = {}
    for f in doc.facts:
        if f.period not in matching:
            continue
        if not f.dimensions:
            plain.setdefault(f.concept, f.value)
        if required_dims and all(f.dimensions.get(ax) == mem
                                 for ax, mem in required_dims.items()):
            scoped.setdefault(f.concept, f.value)

    if required_dims:
        return {**plain, **scoped}   # scoped overrides plain
    else:
        fallback = {}
        for f in doc.facts:
            if f.period not in matching:
                continue
            if f.dimensions:
                fallback.setdefault(f.concept, f.value)
        return {**fallback, **plain} # plain overrides fallback

def _scale_label(scale):
    if scale is None:
        return "as filed"
    s = str(scale)
    return {"3": "thousands", "-3": "thousands",
            "6": "millions",  "-6": "millions",
            "9": "billions",  "-9": "billions"}.get(s, f"scale={s}")

def _common_currency(doc):
    from collections import Counter
    counts = Counter()
    for u in doc.units.values():
        m = u.measure or ""
        if m.startswith("iso4217:"):
            counts[m.split(":", 1)[1]] += 1
    return counts.most_common(1)[0][0] if counts else "USD"

def _label(doc, concept, preferred=None):
    if doc.labs is not None:
        t = doc.labs.get(concept, preferred_label=preferred)
        if t:
            return t
    return concept.split(":", 1)[-1]

def _available_periods(doc, role_short):
    """Return the 2 most recent period_end dates with at least one fact for this role."""
    if doc.pres is None:
        return [doc.filing.period_end] if doc.filing.period_end else []
    concepts = {a.child for a in doc.pres.arcs if a.role_short == role_short}
    concepts |= {a.parent for a in doc.pres.arcs if a.role_short == role_short}
    ends = set()
    for f in doc.facts:
        if f.concept not in concepts:
            continue
        ctx = doc.periods.get(f.period)
        if ctx is None:
            continue
        date = ctx.end if ctx.type == "duration" else ctx.date
        if date:
            ends.add(date)
    return sorted(ends, reverse=True)[:2]

def _build_statement_df(doc, role_short, periods):
    """Return a flat DataFrame with one value column per period_end."""
    if doc.pres is None:
        return pd.DataFrame()
    pres_arcs = [a for a in doc.pres.arcs if a.role_short == role_short]
    if not pres_arcs:
        return pd.DataFrame()

    calc_weight  = {}
    calc_parents = set()
    if doc.calc is not None:
        for a in doc.calc.arcs:
            if a.role_short == role_short:
                calc_weight.setdefault(a.child, a.weight)
                calc_parents.add(a.parent)

    children_by_parent = defaultdict(list)
    is_child = set()
    for a in pres_arcs:
        children_by_parent[a.parent].append(a)
        is_child.add(a.child)
    roots = sorted(set(children_by_parent.keys()) - is_child)

    required_dims = _infer_primary_dims(doc, role_short) or None
    all_values    = {p: _values_at(doc, p, required_dims) for p in periods}

    from collections import Counter
    scales = Counter()
    for c in is_child | set(children_by_parent.keys()):
        for f in doc.facts:
            if f.concept == c and f.scale and not f.dimensions:
                scales[f.scale] += 1
                break
    common_scale = scales.most_common(1)[0][0] if scales else None

    rows = []

    def walk(node, depth, preferred):
        if _is_skip(node):
            for c in sorted(children_by_parent.get(node, []), key=lambda x: x.order):
                walk(c.child, depth, c.preferred_label)
            return

        children = sorted(children_by_parent.get(node, []), key=lambda x: x.order)

        if _is_abstract(node):
            lbl = _label(doc, node, preferred)
            row = {"concept": node, "label": lbl, "depth": depth,
                   "is_header": True, "is_subtotal": False, "sign": "",
                   "scale": common_scale}
            for p in periods:
                row[p] = None
            rows.append(row)
            for c in children:
                walk(c.child, depth + 1, c.preferred_label)
            return

        real_children = [c for c in children
                         if not _is_skip(c.child) and not _is_abstract(c.child)]
        is_leaf     = not real_children
        is_subtotal = node in calc_parents and not is_leaf
        any_value   = any(node in all_values[p] for p in periods)

        w    = calc_weight.get(node)
        sign = ""
        if is_leaf and not is_subtotal and w is not None:
            sign = "+" if w >= 0 else "-"

        lbl = _label(doc, node, preferred)
        row = {"concept": node, "label": lbl, "depth": depth,
               "is_header": not is_leaf and not any_value,
               "is_subtotal": is_subtotal, "sign": sign, "scale": common_scale}
        for p in periods:
            row[p] = all_values[p].get(node)
        rows.append(row)

        for c in children:
            walk(c.child, depth + 1, c.preferred_label)

    for root in roots:
        walk(root, 0, None)

    return pd.DataFrame(rows)

def _df_to_csv_df(df, periods):
    out = df[["concept", "label", "depth", "sign", "scale"] + periods].copy()
    out.insert(1, "display_label", out["depth"].apply(lambda d: "  " * d) + out["label"])
    return out.drop(columns=["label", "depth"])

def _fmt_value(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    if v < 0:
        return f"({abs(v):,.0f})"
    return f"{v:,.0f}"

def _compute_balance_check(doc, role_short, periods, all_values):
    """
    Check Assets == Liabilities + Equity for balance-sheet roles.
    Returns a list of (label, ok) per period, or None if not applicable.
    """
    ASSETS  = ["us-gaap:Assets"]
    LAE     = ["us-gaap:LiabilitiesAndStockholdersEquity",
               "us-gaap:LiabilitiesAndStockholderSEquity"]
    LIAB    = ["us-gaap:Liabilities"]
    EQUITY  = ["us-gaap:StockholdersEquity",
               "us-gaap:StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"]

    def first(vals, keys):
        for k in keys:
            if k in vals:
                return vals[k]
        return None

    results   = []
    has_check = False
    for p in periods:
        vals   = all_values[p]
        assets = first(vals, ASSETS)
        lae    = first(vals, LAE)
        liab   = first(vals, LIAB)
        equity = first(vals, EQUITY)
        if assets is None:
            continue
        if lae is not None:
            ok = abs(assets - lae) < 2
            has_check = True
            results.append((
                f"{'OK' if ok else 'X'} Assets={_fmt_value(assets)} == L&E={_fmt_value(lae)}  [{p}]",
                ok
            ))
        elif liab is not None and equity is not None:
            ok = abs(assets - (liab + equity)) < 2
            has_check = True
            results.append((
                f"{'OK' if ok else 'X'} Assets={_fmt_value(assets)} == Liab({_fmt_value(liab)})+Equity({_fmt_value(equity)})  [{p}]",
                ok
            ))
    return results if has_check else None

def _render_html_table(df, title, periods, scale, currency, balance_check=None):
    scale_txt = _scale_label(scale)
    period_headers = "".join(
        f'<th style="text-align:right; padding:6px 8px; color:#4a90d9; min-width:120px;">{p}</th>'
        for p in periods
    )
    balance_html = ""
    if balance_check:
        items = "  |  ".join(
            f'<span style="color:{"#27ae60" if ok else "#c0392b"};">{label}</span>'
            for label, ok in balance_check
        )
        balance_html = f'<p style="font-size:0.82em; color:#555; margin-bottom:12px;">{items}</p>'

    header_html = f"""
    <div style="font-family: 'Segoe UI', Arial, sans-serif; max-width: {700 + 130*len(periods)}px; margin: 0 auto;">
      <h2 style="color: #1a1a2e; border-bottom: 2px solid #4a90d9; padding-bottom: 8px; margin-bottom: 4px;">
        {title}
      </h2>
      <p style="color: #666; font-size: 0.9em; margin-top: 0; margin-bottom: 6px;">
        in {scale_txt} {currency}
      </p>
      {balance_html}
      <table style="width:100%; border-collapse: collapse; font-size: 0.92em;">
        <thead>
          <tr style="border-bottom: 2px solid #4a90d9;">
            <th style="text-align:left; padding: 6px 4px; color: #4a90d9;">Line item</th>
            {period_headers}
          </tr>
        </thead>
        <tbody>
    """
    rows_html = []
    for _, row in df.iterrows():
        indent = "&nbsp;" * (row["depth"] * 4)
        lbl    = row["label"]
        sign   = row["sign"] or ""

        if row["is_header"]:
            value_cells = "".join("<td></td>" for _ in periods)
            rows_html.append(
                f'<tr style="background:#f0f4fa;">'
                f'<td style="padding:5px 4px; font-weight:600; color:#1a1a2e;">{indent}{lbl}</td>'
                f'{value_cells}</tr>'
            )
        elif row["is_subtotal"]:
            value_cells = ""
            for p in periods:
                val   = _fmt_value(row[p])
                color = "#c0392b" if val.startswith("(") else "#1a1a2e"
                value_cells += (f'<td style="text-align:right; padding:5px 8px; font-weight:700;'
                                f' color:{color}; font-family:monospace;">{val}</td>')
            rows_html.append(
                f'<tr style="border-top: 1px solid #aaa;">'
                f'<td style="padding:5px 4px; font-weight:700; color:#1a1a2e;">{indent}{lbl}</td>'
                f'{value_cells}</tr>'
            )
        else:
            sign_html   = (f'<span style="color:#888; margin-right:4px; font-size:0.85em;">{sign}</span>'
                           if sign else "")
            value_cells = ""
            for p in periods:
                val   = _fmt_value(row[p])
                color = "#c0392b" if val.startswith("(") else "#333"
                value_cells += (f'<td style="text-align:right; padding:4px 8px;'
                                f' color:{color}; font-family:monospace;">{val}</td>')
            rows_html.append(
                f'<tr style="border-bottom: 1px solid #eee;">'
                f'<td style="padding:4px 4px; color:#333;">{indent}{sign_html}{lbl}</td>'
                f'{value_cells}</tr>'
            )

    return header_html + "\n".join(rows_html) + "</tbody></table></div>"


In [11]:
# ── Widget definitions ───────────────────────────────────────────────────────

json_files = sorted(OUTPUT_DIR.glob("*.json"))
file_options = [(f.name, f) for f in json_files]

w_file = widgets.Dropdown(
    options=file_options,
    description="Filing:",
    layout=widgets.Layout(width="520px"),
    style={"description_width": "80px"},
)

w_role = widgets.Dropdown(
    options=[],
    description="Statement:",
    layout=widgets.Layout(width="520px"),
    style={"description_width": "80px"},
)

w_btn_load = widgets.Button(
    description="Load filing",
    button_style="info",
    icon="folder-open",
    layout=widgets.Layout(width="140px"),
)

w_btn_view = widgets.Button(
    description="Visualize",
    button_style="success",
    icon="eye",
    layout=widgets.Layout(width="120px"),
    disabled=True,
)

w_btn_save = widgets.Button(
    description="Save",
    button_style="warning",
    icon="download",
    layout=widgets.Layout(width="100px"),
    disabled=True,
)

w_status = widgets.HTML(value="")
w_out = widgets.Output()

row1 = widgets.HBox([w_file, w_btn_load])
row2 = widgets.HBox([w_role])
row3 = widgets.HBox([w_btn_view, w_btn_save, w_status])

panel = widgets.VBox(
    [row1, row2, row3, w_out],
    layout=widgets.Layout(padding="12px", border="1px solid #ddd", border_radius="8px"),
)


In [12]:
# ── Callbacks ────────────────────────────────────────────────────────────────

def _set_status(msg, color="#555"):
    w_status.value = f'<span style="color:{color}; font-size:0.9em; margin-left:12px;">{msg}</span>'


def on_load(_btn):
    global _doc, _role_map
    path = w_file.value
    if path is None:
        _set_status("No file selected.", "#c0392b")
        return

    _set_status("Loading…")
    w_btn_view.disabled = True
    w_btn_save.disabled = True

    with w_out:
        clear_output(wait=True)

    try:
        _doc = Document.load(path)
    except Exception as e:
        _set_status(f"Error loading: {e}", "#c0392b")
        return

    role_defs = {}
    if _doc.pres is not None:
        for a in _doc.pres.arcs:
            if a.role_short not in role_defs and a.role_definition:
                role_defs[a.role_short] = a.role_definition
    if _doc.calc is not None:
        for rs, rd in _doc.calc.role_definitions.items():
            role_defs.setdefault(rs, rd)

    roles_with_data = {rs for rs in role_defs if _available_periods(_doc, rs)}
    role_defs = {rs: rd for rs, rd in role_defs.items() if rs in roles_with_data}

    primary_keywords = ["income", "operation", "balance", "cash", "equity", "comprehensive"]

    def _sort_key(item):
        rs, rd = item
        is_primary = any(k in rd.lower() for k in primary_keywords)
        return (0 if is_primary else 1, rd)

    sorted_roles = sorted(role_defs.items(), key=_sort_key)
    _role_map = {rs: rd for rs, rd in sorted_roles}

    options = [(f"{rd}  [{rs}]", rs) for rs, rd in sorted_roles]
    w_role.options = options
    w_role.value = options[0][1] if options else None

    f = _doc.filing
    _set_status(
        f"Loaded {f.form or '?'} FY{f.fiscal_year or '?'} · {len(_doc.facts)} facts · {len(options)} statements",
        "#27ae60",
    )
    w_btn_view.disabled = False


def on_visualize(_btn):
    if _doc is None:
        _set_status("Load a filing first.", "#c0392b")
        return

    role_short = w_role.value
    if not role_short:
        _set_status("Select a statement.", "#c0392b")
        return

    periods = _available_periods(_doc, role_short)
    if not periods:
        with w_out:
            clear_output(wait=True)
            print("No data found for this statement.")
        return

    df = _build_statement_df(_doc, role_short, periods)
    if df.empty:
        with w_out:
            clear_output(wait=True)
            print("No data found for this statement.")
        return

    title    = _role_map.get(role_short, role_short)
    scale    = df["scale"].dropna().iloc[0] if not df["scale"].dropna().empty else None
    currency = _common_currency(_doc)

    # Recompute all_values (same required_dims as _build_statement_df used)
    required_dims = _infer_primary_dims(_doc, role_short) or None
    all_values    = {p: _values_at(_doc, p, required_dims) for p in periods}
    balance_check = _compute_balance_check(_doc, role_short, periods, all_values)

    html = _render_html_table(df, title, periods, scale, currency, balance_check)

    with w_out:
        clear_output(wait=True)
        display(HTML(html))

    w_btn_save.disabled = False
    n_ok = sum(1 for _, ok in balance_check if ok) if balance_check else None
    check_msg = f" · balance {'✓' if n_ok == len(balance_check) else '✗'}" if balance_check else ""
    _set_status(f"{len(df)} rows · {len(periods)} period(s){check_msg}", "#27ae60")

    w_btn_save._cached_df          = df
    w_btn_save._cached_html        = html
    w_btn_save._cached_role_short  = role_short
    w_btn_save._cached_periods     = periods
    w_btn_save._cached_title       = title


def on_save(_btn):
    df = getattr(w_btn_save, "_cached_df", None)
    if df is None:
        _set_status("Nothing to save — visualize first.", "#c0392b")
        return

    role_short = w_btn_save._cached_role_short
    periods    = w_btn_save._cached_periods
    title      = w_btn_save._cached_title
    html       = w_btn_save._cached_html

    filing   = _doc.filing
    stem     = Path(filing.source_file).stem if filing.source_file else Path(w_file.value).stem
    safe_role = role_short[:60].replace("/", "_").replace(" ", "_")
    base_name = f"{stem}__{safe_role}"

    save_dir = Path("../data/output/statements")
    save_dir.mkdir(parents=True, exist_ok=True)

    csv_df   = _df_to_csv_df(df, periods)
    csv_path = save_dir / f"{base_name}.csv"
    csv_df.to_csv(csv_path, index=False)

    html_path = save_dir / f"{base_name}.html"
    full_html = f"""<!DOCTYPE html>
<html><head>
<meta charset="utf-8">
<title>{title}</title>
<style>body {{ font-family: 'Segoe UI', Arial, sans-serif; padding: 24px; }}</style>
</head><body>
{html}
</body></html>"""
    html_path.write_text(full_html, encoding="utf-8")

    _set_status(f"Saved → {csv_path.name}  +  {html_path.name}", "#27ae60")
    with w_out:
        display(HTML(
            f'<p style="font-size:0.85em; color:#555;">'
            f'Saved to <code>{save_dir.resolve()}</code><br>'
            f'&nbsp; · {csv_path.name}<br>'
            f'&nbsp; · {html_path.name}</p>'
        ))


w_btn_load.on_click(on_load)
w_btn_view.on_click(on_visualize)
w_btn_save.on_click(on_save)


In [13]:
display(panel)